In [1]:
from pathlib import Path

import pandas as pd

In [2]:
pwd = Path.cwd()

In [3]:
hgt = pwd.parent / "hordeum_panicoid_hgt" 
annot_dir = hgt / "annot"
orthogroup_dir = hgt / "positive_protein_coding" / "orthogroup_clusters"

# Preparing Eggnog mapper output for enrichment analysis

In [5]:
ortho_dir = annot_dir / "orthogroups" 
eggnog_go = pd.read_csv(ortho_dir / "all_groups_eggnog_mapper.tsv", sep='\t')
eggnog_go.head()

,#query,seed_ortholog,evalue,score,eggNOG_OGs,max_annot_lvl,COG_category,Description,Preferred_name,GOs,...,KEGG_ko,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs
0,HMARINUM.BCC2001.r1.2HG00091550.1:0-1269,4513.MLOC_70630.1,7.730000e-207,587.0,"COG2124@1|root,KOG0156@2759|Eukaryota,37QA7@33...",35493|Streptophyta,Q,Belongs to the cytochrome P450 family,-,-,...,ko:K00512,"ko00140,ko01100,ko04913,ko04917,ko04927,ko0493...","M00109,M00110","R02211,R03783,R04852,R04853,R08517,R08518","RC00607,RC00660,RC00923,RC01222","ko00000,ko00001,ko00002,ko00199,ko01000",-,-,-,p450
1,HMARINUM.BCC2001.r1.2HG00091580.1:0-441,37682.EMT09706,3.090000e-103,298.0,"2E0DQ@1|root,2S7UD@2759|Eukaryota,37UTH@33090|...",35493|Streptophyta,S,PLAC8 family,-,-,...,-,-,-,-,-,-,-,-,-,PLAC8
2,HMARINUM.BCC2001.r1.2HG00091590.1:0-1584,4513.MLOC_49936.1,0.000000e+00,967.0,"COG0277@1|root,2QQWK@2759|Eukaryota,37QU2@3309...",35493|Streptophyta,C,Belongs to the oxygen-dependent FAD-linked oxi...,-,-,...,-,-,-,-,-,-,-,-,-,"BBE,FAD_binding_4"
3,HMARINUM.BCC2001.r1.2HG00091600.2:0-1593,4513.MLOC_77123.1,0.000000e+00,984.0,"COG0277@1|root,2QQWK@2759|Eukaryota,37QU2@3309...",35493|Streptophyta,C,Belongs to the oxygen-dependent FAD-linked oxi...,-,-,...,-,-,-,-,-,-,-,-,-,"BBE,FAD_binding_4"
4,HMARINUM.BCC2001.r1.2HG00091610.2:0-663,4538.ORGLA06G0215700.1,8.710000e-73,227.0,"KOG1075@1|root,KOG1075@2759|Eukaryota,384BG@33...",35493|Streptophyta,S,Reverse transcriptase-like,-,-,...,-,-,-,-,-,-,-,-,-,RVT_3


In [7]:
eggnog_go.columns

Index(['#query', 'seed_ortholog', 'evalue', 'score', 'eggNOG_OGs',
       'max_annot_lvl', 'COG_category', 'Description', 'Preferred_name', 'GOs',
       'EC', 'KEGG_ko', 'KEGG_Pathway', 'KEGG_Module', 'KEGG_Reaction',
       'KEGG_rclass', 'BRITE', 'KEGG_TC', 'CAZy', 'BiGG_Reaction', 'PFAMs'],
      dtype='object')

### First some clean-up: rename columns, delete metadata rows (leftovers from concatenating output files) etc.

In [6]:
eggnog_go = eggnog_go.rename(columns={'#query': 'gene', 'GOs': 'go'})

In [7]:
eggnog_go.gene = eggnog_go.gene.apply(lambda x: x.split(':')[0])

In [13]:
eggnog_go.head()

,gene,seed_ortholog,evalue,score,eggNOG_OGs,max_annot_lvl,COG_category,Description,Preferred_name,go,...,KEGG_ko,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs
0,HMARINUM.BCC2001.r1.2HG00091550.1,4513.MLOC_70630.1,7.730000e-207,587.0,"COG2124@1|root,KOG0156@2759|Eukaryota,37QA7@33...",35493|Streptophyta,Q,Belongs to the cytochrome P450 family,-,-,...,ko:K00512,"ko00140,ko01100,ko04913,ko04917,ko04927,ko0493...","M00109,M00110","R02211,R03783,R04852,R04853,R08517,R08518","RC00607,RC00660,RC00923,RC01222","ko00000,ko00001,ko00002,ko00199,ko01000",-,-,-,p450
1,HMARINUM.BCC2001.r1.2HG00091580.1,37682.EMT09706,3.090000e-103,298.0,"2E0DQ@1|root,2S7UD@2759|Eukaryota,37UTH@33090|...",35493|Streptophyta,S,PLAC8 family,-,-,...,-,-,-,-,-,-,-,-,-,PLAC8
2,HMARINUM.BCC2001.r1.2HG00091590.1,4513.MLOC_49936.1,0.000000e+00,967.0,"COG0277@1|root,2QQWK@2759|Eukaryota,37QU2@3309...",35493|Streptophyta,C,Belongs to the oxygen-dependent FAD-linked oxi...,-,-,...,-,-,-,-,-,-,-,-,-,"BBE,FAD_binding_4"
3,HMARINUM.BCC2001.r1.2HG00091600.2,4513.MLOC_77123.1,0.000000e+00,984.0,"COG0277@1|root,2QQWK@2759|Eukaryota,37QU2@3309...",35493|Streptophyta,C,Belongs to the oxygen-dependent FAD-linked oxi...,-,-,...,-,-,-,-,-,-,-,-,-,"BBE,FAD_binding_4"
4,HMARINUM.BCC2001.r1.2HG00091610.2,4538.ORGLA06G0215700.1,8.710000e-73,227.0,"KOG1075@1|root,KOG1075@2759|Eukaryota,384BG@33...",35493|Streptophyta,S,Reverse transcriptase-like,-,-,...,-,-,-,-,-,-,-,-,-,RVT_3


In [28]:
eggnog_go[eggnog_go.duplicated()]

,gene,seed_ortholog,evalue,score,eggNOG_OGs,max_annot_lvl,COG_category,Description,Preferred_name,go,...,KEGG_ko,KEGG_Pathway,KEGG_Module,KEGG_Reaction,KEGG_rclass,BRITE,KEGG_TC,CAZy,BiGG_Reaction,PFAMs
92685,## Total time (seconds),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92686,## Rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137509,## Total time (seconds),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137510,## Rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181073,## Total time (seconds),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181074,## Rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
203398,## Total time (seconds),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
203399,## Rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
248010,## Total time (seconds),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
248011,## Rate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
eggnog_go = eggnog_go.drop_duplicates()
eggnog_go.shape

(925590, 21)

In [9]:
eggnog_go = eggnog_go[eggnog_go.gene.str.startswith('H')]

In [10]:
eggnog_go_ess = eggnog_go[["gene", "go"]]
eggnog_go_ess.head()

,gene,go
0,HMARINUM.BCC2001.r1.2HG00091550.1,-
1,HMARINUM.BCC2001.r1.2HG00091580.1,-
2,HMARINUM.BCC2001.r1.2HG00091590.1,-
3,HMARINUM.BCC2001.r1.2HG00091600.2,-
4,HMARINUM.BCC2001.r1.2HG00091610.2,-


In [17]:
eggnog_groups = pd.merge(groups, eggnog_go_ess, on="gene", how="left")
eggnog_groups.shape

(1027577, 3)

In [35]:
groups[~groups.gene.isin(eggnog_go.gene)].shape
# pro ne asi nebyl vystup, prip. zkusit znovu

(102010, 2)

In [48]:
eggnog_groups.go.str.split(',')

0                                                        NaN
1                                                        NaN
2                                                        NaN
3                                                        [-]
4                                                        NaN
                                 ...                        
1027572                                                  NaN
1027573                                                  [-]
1027574    [GO:0003674, GO:0003824, GO:0003993, GO:000462...
1027575                                                  NaN
1027576                                                  NaN
Name: go, Length: 1027577, dtype: object

In [18]:
eggnog_groups = eggnog_groups.fillna('-')
eggnog_groups.go = eggnog_groups.go.str.split(',')

In [19]:
eggnog_groups.columns

Index(['group', 'gene', 'go'], dtype='object')

### And again aggregate gene-GO term into orthogroup-GO term

In [37]:
def aggregate_to_set(series):
    # Flatten lists and convert to set
    return set(item for sublist in series for item in sublist)

In [38]:
eggnog_grouped = eggnog_groups.groupby("group").agg({"gene": lambda x: set(x), "go": aggregate_to_set}).reset_index()

In [39]:
eggnog_grouped.head(10)

,group,gene,go
0,N0.HOG0000000,"{HORVU.MOREX.PROJ.3HG00275700.1, HORVU.MOREX.P...",{-}
1,N0.HOG0000001,"{HCHILENSE.GRA1000.r1.5HG00603760.1, HINTERCED...",{-}
2,N0.HOG0000002,"{HGUSSONEANUM.BCC2005.r1.3HG00302330.1, HSTENO...",{-}
3,N0.HOG0000003,"{HBOGDANII.H240.r1.2HG00246270.1, HBREVISUBULA...",{-}
4,N0.HOG0000004,{HBULBOSUM.FB19_011_3.PROJ.r1.chr6H_2G01846030...,{-}
5,N0.HOG0000005,{HBULBOSUM.FB19_011_3.PROJ.r1.chr1H_2G01729600...,{-}
6,N0.HOG0000006,"{HSECALINUM.BCC2004.r1.1H_2G00043710.1, HBOGDA...",{-}
7,N0.HOG0000007,"{HJUBATUM.BCC2055.r1.1H_2G00055180.1, HJUBATUM...",{-}
8,N0.HOG0000008,"{HJUBATUM.BCC2055.r1.2H_2G00143720.1, HJUBATUM...",{-}
9,N0.HOG0000009,"{HCOMOSUM.NGB18417.r1.1HG00079050.1, HPUBIFLOR...",{-}


In [40]:
eggnog_grouped.shape

(74977, 3)

### Then reformat into orthogroup-GO term (one per line aka exploding)

In [41]:
exp_eggnog_go = eggnog_grouped.explode('go', ignore_index=True)

In [42]:
exp_eggnog_go.head(10)

,group,gene,go
0,N0.HOG0000000,"{HORVU.MOREX.PROJ.3HG00275700.1, HORVU.MOREX.P...",-
1,N0.HOG0000001,"{HCHILENSE.GRA1000.r1.5HG00603760.1, HINTERCED...",-
2,N0.HOG0000002,"{HGUSSONEANUM.BCC2005.r1.3HG00302330.1, HSTENO...",-
3,N0.HOG0000003,"{HBOGDANII.H240.r1.2HG00246270.1, HBREVISUBULA...",-
4,N0.HOG0000004,{HBULBOSUM.FB19_011_3.PROJ.r1.chr6H_2G01846030...,-
5,N0.HOG0000005,{HBULBOSUM.FB19_011_3.PROJ.r1.chr1H_2G01729600...,-
6,N0.HOG0000006,"{HSECALINUM.BCC2004.r1.1H_2G00043710.1, HBOGDA...",-
7,N0.HOG0000007,"{HJUBATUM.BCC2055.r1.1H_2G00055180.1, HJUBATUM...",-
8,N0.HOG0000008,"{HJUBATUM.BCC2055.r1.2H_2G00143720.1, HJUBATUM...",-
9,N0.HOG0000009,"{HCOMOSUM.NGB18417.r1.1HG00079050.1, HPUBIFLOR...",-


In [45]:
exp_eggnog_go[["go", "group"]].to_csv(ortho_dir / "all_groups_eggnog_go_exploded", sep='\t', index=False)

In [43]:
exp_eggnog_go.go.value_counts()

go
-             55389
GO:0005575    18531
GO:0005623    17946
GO:0044464    17945
GO:0008150    17599
              ...  
GO:0006185        1
GO:0009180        1
GO:0009183        1
GO:0009182        1
GO:0010398        1
Name: count, Length: 12717, dtype: int64